# 02 — Limpieza y Transformación de Datos
**Criterio 3.1.5.I.2 (15%):** Perfilado, reglas de calidad, justificación de tratamientos y reproducibilidad.

Los datos en estado RAW (antes de cualquier limpieza) están en:
`datos para evaluacion/datos_originales/`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.1f}".format)

BASE = Path("../01_datos/procesados")
COLS = ["TipoCompra","codigoOC","Anio","Mes","MontoNetoItemCLP",
        "Region","Servicio","Subtitulo","ProveedorRUT","CodBIP","NombreProyecto","Proy_Gasto_MM"]

print("Cargando datos...")
it  = pd.read_csv(BASE / "compras_presupuesto_items.csv", sep=";", low_memory=False, usecols=COLS)
res = pd.read_csv(BASE / "compras_presupuesto_resumen.csv", sep=";", encoding="utf-8-sig")
pres= pd.read_csv(BASE / "presupuesto_2018_2019_unificado.csv", sep=";", encoding="utf-8-sig")

it  = it[it["MontoNetoItemCLP"] > 0].copy()
it["MM"] = it["MontoNetoItemCLP"] / 1e6
it["Trimestre"] = ((it["Mes"]-1)//3+1).map({1:"Q1",2:"Q2",3:"Q3",4:"Q4"})
it["EsInversion"] = (it["Subtitulo"].fillna(-1).astype(int) == 31).astype(int)
it["TieneBIP"] = it["CodBIP"].notna().astype(int)

print(f"Items  : {len(it):,} filas")
print(f"Resumen: {len(res):,} filas")
print(f"Presup.: {len(pres):,} filas")

Cargando datos...
Items  : 172,310 filas
Resumen: 357 filas
Presup.: 3,179 filas


## 2.1 Vista del Estado RAW

In [2]:
print("=== Primeras filas del dataset crudo ===")
display(it.head(3))
print(f"\nForma original: {it.shape}")
print(f"Tipos de datos:\n{it.dtypes}")

=== Primeras filas del dataset crudo ===


,TipoCompra,codigoOC,ProveedorRUT,MontoNetoItemCLP,Anio,Mes,Region,Servicio,Subtitulo,CodBIP,NombreProyecto,Proy_Gasto_MM,MM,Trimestre,EsInversion,TieneBIP
0,Licitacion,823-39-SE18,52.003.350-5,"34,331,000.0",2019,1,O'Higgins,Dirección de Arquitectura,31.0,NaN,NaN,NaN,34.3,Q1,1,0
1,Licitacion,829-41-SE18,79.747.870-9,"775,252,299.0",2019,1,Magallanes,Dirección de Arquitectura,NaN,NaN,NaN,NaN,775.3,Q1,0,0
2,Licitacion,829-78-SE18,76.057.259-4,"214,800,000.0",2019,1,Magallanes,Dirección de Arquitectura,31.0,NaN,NaN,NaN,214.8,Q1,1,0



Forma original: (172310, 16)
Tipos de datos:
TipoCompra              str
codigoOC                str
ProveedorRUT            str
MontoNetoItemCLP    float64
Anio                  int64
Mes                   int64
Region                  str
Servicio                str
Subtitulo           float64
CodBIP              float64
NombreProyecto          str
Proy_Gasto_MM       float64
MM                  float64
Trimestre               str
EsInversion           int64
TieneBIP              int64
dtype: object


## 2.2 Reglas de Limpieza Aplicadas

In [3]:
reglas = []

# R1: Monto <= 0
n_antes = len(it)
it_c = it[it["MontoNetoItemCLP"] > 0].copy()
n_R1 = n_antes - len(it_c)
reglas.append({"Regla":"R1 - Eliminar monto <= 0","Eliminados":n_R1,
               "Justificacion":"Montos nulos o negativos son errores de carga, no transacciones reales"})

# R2: Cast tipos
it_c["Anio"] = it_c["Anio"].astype(int)
it_c["Mes"]  = it_c["Mes"].astype(int)
reglas.append({"Regla":"R2 - Cast Anio, Mes a int","Eliminados":0,
               "Justificacion":"Consistencia de tipos para agrupaciones y joins"})

# R3: Subtitulo nulo
n_nulo = it_c["Subtitulo"].isna().sum()
it_c["Subtitulo"] = it_c["Subtitulo"].fillna(-1).astype(int)
reglas.append({"Regla":"R3 - Subtitulo nulo -> -1","Eliminados":0,
               "Justificacion":f"{n_nulo:,} items sin clasificacion presupuestaria, se mantienen con etiqueta -1"})

pd_reg = pd.DataFrame(reglas)
display(pd_reg.set_index("Regla").style.set_caption("Tabla 6. Registro de reglas de limpieza"))
print(f"\nDataset limpio: {len(it_c):,} filas ({100*len(it_c)/n_antes:.2f}% del original)")

,Eliminados,Justificacion
Regla,,
R1 - Eliminar monto <= 0,0,"Montos nulos o negativos son errores de carga, no transacciones reales"
"R2 - Cast Anio, Mes a int",0,Consistencia de tipos para agrupaciones y joins
R3 - Subtitulo nulo -> -1,0,"112,451 items sin clasificacion presupuestaria, se mantienen con etiqueta -1"



Dataset limpio: 172,310 filas (100.00% del original)


## 2.3 Variables Derivadas (Feature Engineering)

In [ ]:
# Monto en millones
it_c["MM"] = it_c["MontoNetoItemCLP"] / 1e6

# Monto total por OC
oc_total = it_c.groupby("codigoOC")["MontoNetoItemCLP"].sum().rename("MontoOC_CLP")
it_c = it_c.merge(oc_total, on="codigoOC", how="left")

# Cuartil de monto por OC
it_c["CuartilMonto"] = pd.qcut(it_c["MontoOC_CLP"], q=4,
                                labels=["Q1-Bajo","Q2-MedBajo","Q3-MedAlto","Q4-Alto"])

# Trimestre
it_c["Trimestre"] = ((it_c["Mes"]-1)//3+1).map({1:"Q1",2:"Q2",3:"Q3",4:"Q4"})

# Es inversión (Subtítulo 31)
it_c["EsInversion"] = (it_c["Subtitulo"] == 31).astype(int)

# Tiene código BIP
it_c["TieneBIP"] = it_c["CodBIP"].notna().astype(int)

resumen_vars = pd.DataFrame({
    "Variable"      : ["MM","MontoOC_CLP","CuartilMonto","Trimestre","EsInversion","TieneBIP"],
    "Tipo"          : ["Numérica","Numérica","Ordinal","Nominal","Binaria","Binaria"],
    "Descripcion"   : ["Monto en millones de CLP","Monto total de la OC","Segmento por tamaño económico",
                       "Q1-Q4 agrupación temporal","1 si Subtítulo=31 (inversión)","1 si tiene código BIP"],
    "Justificacion" : ["Escala legible para análisis","Análisis a nivel OC, no ítem",
                       "Segmentación sin distribución supuesta","Estacionalidad sin 12 categorías",
                       "Distingue compras de inversión vs operación","Trazabilidad de compras a proyecto"]
})
display(resumen_vars.set_index("Variable").style.set_caption("Tabla 7. Variables derivadas creadas"))

## 2.4 Verificación Post-Limpieza

In [ ]:
print("Verificacion post-limpieza:")
print(f"  Anios presentes  : {sorted(it_c.Anio.unique())}")
print(f"  Meses presentes  : {sorted(it_c.Mes.unique())}")
print(f"  Tipos de compra  : {sorted(it_c.TipoCompra.unique())}")
print(f"  Nulos en MM      : {it_c.MM.isna().sum()}")
print(f"  Negativos en MM  : {(it_c.MM <= 0).sum()}")
print(f"\nDistribucion CuartilMonto:")
print(it_c.CuartilMonto.value_counts().sort_index())
print(f"\nEsInversion: {it_c.EsInversion.sum():,} items ({100*it_c.EsInversion.mean():.1f}%)")
print(f"TieneBIP   : {it_c.TieneBIP.sum():,} items ({100*it_c.TieneBIP.mean():.1f}%)")